# Nexus cycle attribution (v2)

Logic lives in `src/v2/scripts/` — this notebook is the driver.

Constants: `scripts/config.toml`.


In [ ]:
import sys
from datetime import date
from pathlib import Path

import pandas as pd

# make scripts importable when the kernel cwd is anywhere under the repo
SCRIPTS = Path('/home/ajain/apeksha/pnl_attribution/src/v2/scripts')
ROOT = SCRIPTS.parent  # src/v2
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts import CONFIG, get_ch_client
from scripts.day_attribution import attribute_all_books_day, attribute_book_day

client = get_ch_client()
print('books =', len(CONFIG.book_names))
print('contract size =', CONFIG.contract.size, '| clearing =', CONFIG.contract.clearing_rate)


books = 10
contract size = 1000 | clearing = 0.0019


### Single book / all days

Day-total **value_added c/bbl** by strategy for one book (`c/bbl` slippage sign: + = cost, − = gain).
Volume = Σ |position_received| per cycle, summed over the day.

In [3]:
BOOK = 'Hedger Spreads'
START = date(2026, 9, 17)
END = date.today()  # through today

dates = pd.date_range(START, END, freq='D').date.tolist()
rows = []
for d in dates:
    print(f'=== {d} {BOOK} ===')
    one = attribute_book_day(client, d, BOOK, progress_every=0)
    if one.empty or 'DAY TOTAL' not in one.index:
        print('  (no cycles)')
        continue
    day = one.loc['DAY TOTAL']
    abs_lots = float(day[('meta', 'abs_lots', '')])
    rec = {'date': d, 'abs_lots': abs_lots}
    for stage in CONFIG.stage_order:
        rec[stage] = float(day[('c/bbl', stage, 'value_added_by_strategy')])
    rows.append(rec)

book_all_days = pd.DataFrame(rows).set_index('date')
book_all_days.round(4)

=== 2026-09-17 Hedger Spreads ===
  Hedger Spreads: prefetching market data (476 cycles)…
=== 2026-09-18 Hedger Spreads ===
  Hedger Spreads: prefetching market data (446 cycles)…
=== 2026-09-19 Hedger Spreads ===
  (no cycles)
=== 2026-09-20 Hedger Spreads ===
  (no cycles)
=== 2026-09-21 Hedger Spreads ===
  Hedger Spreads: prefetching market data (462 cycles)…
=== 2026-09-22 Hedger Spreads ===
  Hedger Spreads: prefetching market data (428 cycles)…
=== 2026-09-23 Hedger Spreads ===
  Hedger Spreads: prefetching market data (361 cycles)…
=== 2026-09-24 Hedger Spreads ===
  Hedger Spreads: prefetching market data (351 cycles)…
=== 2026-09-25 Hedger Spreads ===
  Hedger Spreads: prefetching market data (243 cycles)…
=== 2026-09-26 Hedger Spreads ===
  (no cycles)
=== 2026-09-27 Hedger Spreads ===
  (no cycles)
=== 2026-09-28 Hedger Spreads ===
  Hedger Spreads: prefetching market data (326 cycles)…
=== 2026-09-29 Hedger Spreads ===
  Hedger Spreads: prefetching market data (177 cycles)

,abs_lots,TOTAL,transfer_pred,pca,routing
date,,,,,
2026-09-17,68517.0,-0.1329,-0.0,-0.1316,-0.0013
2026-09-18,23179.0,-0.0277,-0.0,-0.0278,0.0001
2026-09-21,40422.0,-0.0329,-0.0,-0.0313,-0.0016
2026-09-22,28697.0,-0.0985,-0.0,-0.0938,-0.0047
2026-09-23,21765.0,-0.0307,-0.0,-0.0321,0.0014
2026-09-24,160487.0,-0.1036,-0.0,-0.0178,-0.0858
2026-09-25,163125.0,-0.0520,-0.0,-0.0084,-0.0436
2026-09-28,100196.0,-0.0098,-0.0,-0.0094,-0.0004
2026-09-29,45680.0,0.0023,-0.0,0.0023,-0.0000


### PCA market risk / all days

Day-total **held_market_risk** for the `pca` stage only — `$` (+ = MTM gain) and `c/bbl` (slippage sign: + = cost, − = gain).

In [5]:
BOOK = 'Hedger Spreads'
START = date(2026, 9, 17)
END = date.today()

dates = pd.date_range(START, END, freq='D').date.tolist()
rows = []
for d in dates:
    print(f'=== {d} {BOOK} pca MR ===')
    one = attribute_book_day(client, d, BOOK, progress_every=0)
    if one.empty or 'DAY TOTAL' not in one.index:
        print('  (no cycles)')
        continue
    day = one.loc['DAY TOTAL']
    abs_lots = float(day[('meta', 'abs_lots', '')])
    rows.append({
        'date': d,
        'abs_lots': abs_lots,
        'pca_mr_$': float(day[('$', 'pca', 'held_market_risk')]),
        'pca_mr_c/bbl': float(day[('c/bbl', 'pca', 'held_market_risk')]),
    })

pca_mr_all_days = pd.DataFrame(rows).set_index('date')
pca_mr_all_days.round(4)

=== 2026-09-17 Hedger Spreads pca MR ===
  Hedger Spreads: prefetching market data (476 cycles)…
=== 2026-09-18 Hedger Spreads pca MR ===
  Hedger Spreads: prefetching market data (446 cycles)…
=== 2026-09-19 Hedger Spreads pca MR ===
  (no cycles)
=== 2026-09-20 Hedger Spreads pca MR ===
  (no cycles)
=== 2026-09-21 Hedger Spreads pca MR ===
  Hedger Spreads: prefetching market data (462 cycles)…
=== 2026-09-22 Hedger Spreads pca MR ===
  Hedger Spreads: prefetching market data (428 cycles)…
=== 2026-09-23 Hedger Spreads pca MR ===
  Hedger Spreads: prefetching market data (361 cycles)…
=== 2026-09-24 Hedger Spreads pca MR ===
  Hedger Spreads: prefetching market data (351 cycles)…
=== 2026-09-25 Hedger Spreads pca MR ===
  Hedger Spreads: prefetching market data (243 cycles)…
=== 2026-09-26 Hedger Spreads pca MR ===
  (no cycles)
=== 2026-09-27 Hedger Spreads pca MR ===
  (no cycles)
=== 2026-09-28 Hedger Spreads pca MR ===
  Hedger Spreads: prefetching market data (326 cycles)…
=== 

,abs_lots,pca_mr_$,pca_mr_c/bbl
date,,,
2026-09-17,68517.0,5585.19,-0.0082
2026-09-18,23179.0,4074.91,-0.0176
2026-09-21,40422.0,4930.81,-0.0122
2026-09-22,28697.0,-7637.73,0.0266
2026-09-23,21765.0,4336.16,-0.0199
2026-09-24,160487.0,-21737.02,0.0135
2026-09-25,163125.0,-4202.54,0.0026
2026-09-28,100196.0,3761.67,-0.0038
2026-09-29,47940.0,1945.97,-0.0041
